# Soft Actor Critic (SAC) Orbital Attitude Control Training

In [1]:
import sys

In [ ]:
!pip install stable_baselines3 optuna

In [22]:
import yaml
import gymnasium as gym
from gymnasium.envs.registration import register
import os
import torch
from omegaconf import OmegaConf
from stable_baselines3.common.env_checker import check_env
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.callbacks import CallbackList
import numpy as np
import optuna
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Google Collab Mounting

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/OrbitalAttitudeAgent/models"
except ImportError:
    # Keep relative checkpoint directory <- not in google drive
    SAVE_DIR = "/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
%cd /content/drive/MyDrive/OrbitalAttitudeAgent

/content/drive/MyDrive/OrbitalAttitudeAgent


In [18]:
from src.envs.utils import(
    random_unit_quaternion,
    omega_matrix,
    quaternion_error,
    rotate_vector,
    random_unit_vector,
    rotate_around_axis,
    to_obs
)

# Gym ENV registration

In [7]:
from src.envs.attitude_env import AttitudeEnv

In [ ]:
register (
    id = "AttitudeControl-v0",
    entry_point = "src.envs.attitude_env:AttitudeEnv"
)
agent_config = OmegaConf.load("/content/drive/MyDrive/OrbitalAttitudeAgent/agent.yaml")
env = AttitudeEnv(agent_config['env'])

In [9]:
print(agent_config['env'])

{'alpha': 0.5, 'beta': 0.01, 'tau_max': 0.1, 'inertia_tensor': [0.01, 0.01, 0.02], 'dt': 0.1, 'max_steps': 500, 'q_target': [1.0, 0.0, 0.0, 0.0], 'success_threshold': 2.0, 'success_bonus': 10.0, 'f_zone_half_angle': 20.0, 'boresight_axis': [1.0, 0.0, 0.0], 'orbital_rate': 0.00116, 'sun_vector_init': [1.0, 0.0, 0.0]}


In [10]:
check_env(env)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/env_checker.py:515: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(


# Training

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/OrbitalAttitudeAgent/models"

In [ ]:
def evaluate_success_rate(model, env, n_episodes=100):
    successes = 0
    for _ in range(n_episodes):
        obs, _ = env.reset()
        for _ in range(env.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, terminated, _, _ = env.step(action)
            if terminated:
                # check if terminated via success not fzone
                q_error = quaternion_error(env.q, env.q_target)
                angle_error = 2 * np.arccos(np.abs(q_error[0]))
                if angle_error < env.success_threshold:
                    successes += 1
                break
    return successes / n_episodes

In [ ]:
class MetricsCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.timesteps = []
        self.mean_rewards = []
        self.mean_ep_lengths = []
        self.success_rates = []
        self.entropies = []

    def _on_step(self) -> bool:
        if self.num_timesteps % 10_000 == 0:
            rate = evaluate_success_rate(self.model, self.training_env)
            self.success_rates.append(rate)
        return True

    def _on_rollout_end(self) -> None:
        # called at the end of each rollout collection
        # episode metrics are available here
        if len(self.model.ep_info_buffer) > 0:
            ep_rewards = [ep['r'] for ep in self.model.ep_info_buffer]
            ep_lengths = [ep['l'] for ep in self.model.ep_info_buffer]
            
            self.timesteps.append(self.num_timesteps)
            self.mean_rewards.append(np.mean(ep_rewards))
            self.mean_ep_lengths.append(np.mean(ep_lengths))
            
            if hasattr(self.model, 'ent_coef_tensor'):
                self.entropies.append(self.model.ent_coef_tensor.item())

In [ ]:
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
    gamma = trial.suggest_float("gamma", 0.95, 0.999)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
    success_bonus = trial.suggest_float("success_bonus", 5.0, 20.0)

    # create a fresh config copy
    trial_cfg = OmegaConf.create({
        **OmegaConf.to_container(agent_config.env, resolve=True),
        "success_bonus": success_bonus
    })
    trial_env = AttitudeEnv(trial_cfg)

    model = SAC(
        "MlpPolicy",
        trial_env,                       
        learning_rate=learning_rate,
        gamma=gamma,
        batch_size=batch_size,
        verbose=0
    )
    model.learn(total_timesteps=100_000)
    mean_reward, _ = evaluate_policy(model, trial_env, n_eval_episodes=20)
    return mean_reward

In [ ]:
storage = "sqlite:////content/drive/MyDrive/OrbitalAttitudeAgent/optuna.db"
study = optuna.create_study(
    direction="maximize",
    storage=storage,
    study_name="sac_attitude_sweep",
    load_if_exists=True 
)

study.optimize(objective, n_trials=20)
best_params = study.best_params

print("Best params:", best_params)
print("Best reward:", study.best_value)

In [ ]:
model = SAC(
    "MlpPolicy",
    env,
    device = "cuda",
    learning_rate=best_params["learning_rate"],
    buffer_size = agent_config.sac.buffer_size,
    tau = agent_config.sac.tau, #polyak averaging coefficient
    gamma=best_params["gamma"], #discount factor
    batch_size=best_params["batch_size"],
    train_freq = agent_config.sac.train_freq,
    gradient_steps = agent_config.sac.gradient_steps,
    verbose = 1,
    tensorboard_log = "./logs/"
)

In [ ]:
print(model.device)

In [ ]:
checkpoint_callback = CheckpointCallback(
    save_freq = 50000,
    save_path = f"{SAVE_DIR}/checkpoints",
    name_prefix = "sac_attitude"
)
metrics_cb = MetricsCallback()
model.learn(
    total_timesteps = agent_config.sac.total_timesteps,
    callback = CallbackList(checkpoint_callback,metrics_cb)
    )

In [ ]:
model.save(f"{SAVE_DIR}/sac_attitude_final")

# Plot Metrics

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

In [ ]:
timesteps = metrics_cb.timesteps
rewards = metrics_cb.mean_rewards
ep_lengths = metrics_cb.mean_ep_lengths

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig)

# Mean episode reward
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(metrics_cb.timesteps, metrics_cb.mean_rewards)
ax1.set_title("Mean Episode Reward")
ax1.set_xlabel("Timesteps")
ax1.set_ylabel("Reward")

# Mean episode length 
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(metrics_cb.timesteps, metrics_cb.mean_ep_lengths)
ax2.set_title("Mean Episode Length")
ax2.set_xlabel("Timesteps")
ax2.set_ylabel("Steps")

# Success rate
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(metrics_cb.timesteps, metrics_cb.success_rates)
ax3.axhline(y=0.97, color='r', linestyle='--', label='Yang et al. baseline')
ax3.set_title("Success Rate")
ax3.set_xlabel("Timesteps")
ax3.set_ylabel("Rate")
ax3.legend()

# Entropy coefficient
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(metrics_cb.timesteps, metrics_cb.entropies)
ax4.set_title("Entropy (Exploration)")
ax4.set_xlabel("Timesteps")
ax4.set_ylabel("α * H")

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/OrbitalAttitudeAgent/training_curves.png')
plt.show()